# Adult literacy and global D1

This notebook applies the country-level analysis used for electricity, water and internet to adult literacy: WDI indicator `WB_WDI_SE_ADT_LITR_ZS`.

A low-literacy country is defined here as one where **fewer than 50% of adults ages 15+ are literate**. This measures where decile members live; it does not identify which individuals are literate.

Literacy observations come from censuses and surveys, not an annual measurement system. The notebook therefore separates a recent snapshot using each country's latest observation up to 2024 from exact-year results with their coverage shown.

In [1]:
import importlib
import sys
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / 'indicator_analysis_helpers.py').exists() else Path.cwd() / 'dev_indicators'
sys.path.insert(0, str(NOTEBOOK_DIR))
import indicator_analysis_helpers
importlib.reload(indicator_analysis_helpers)
from indicator_analysis_helpers import D1_FILE, REGION_COLORS, fetch_indicator, load_d1_shares, run_indicator_analysis

INDICATOR_YEAR = 2024
LOW_LITERACY_THRESHOLD = 50
analysis = run_indicator_analysis(
    'WB_WDI',
    'WB_WDI_SE_ADT_LITR_ZS',
    'coverage_pct',
    low_coverage_min_countries=40,
)

D1_PATH = D1_FILE if D1_FILE.exists() else NOTEBOOK_DIR.parent / D1_FILE.name
distribution = pd.read_csv(D1_PATH)
for column in ('year', 'welf', 'pop'):
    distribution[column] = pd.to_numeric(distribution[column], errors='coerce')
distribution = distribution.dropna(subset=['year', 'welf', 'pop'])

global_distribution = distribution[distribution['year'].eq(INDICATOR_YEAR)].sort_values('welf', kind='stable').copy()
global_population = global_distribution['pop'].sum()
global_distribution['decile'] = np.minimum(
    np.ceil(global_distribution['pop'].cumsum() / (global_population / 10)),
    10,
).astype(int)
decile_distribution = (
    global_distribution.groupby(['code', 'decile'], as_index=False)
    .agg(decile_population=('pop', 'sum'))
)

country_latest = (
    fetch_indicator('WB_WDI', 'WB_WDI_SE_ADT_LITR_ZS')
    .query('year <= @INDICATOR_YEAR')
    .sort_values('year')
    .drop_duplicates('code', keep='last')
    .rename(columns={'year': 'observation_year', 'indicator_value': 'adult_literacy_pct'})
)
d1_2024 = load_d1_shares().query('year == @INDICATOR_YEAR').copy()
snapshot = d1_2024.merge(
    country_latest[['code', 'observation_year', 'adult_literacy_pct', 'region']],
    on='code',
    how='left',
)
snapshot['observation_age'] = INDICATOR_YEAR - snapshot['observation_year']
snapshot['low_literacy_country'] = snapshot['adult_literacy_pct'].lt(LOW_LITERACY_THRESHOLD)

observed_snapshot = snapshot.dropna(subset=['adult_literacy_pct']).copy()
d1_coverage = observed_snapshot['d1_population_share'].sum()
d1_low_share = observed_snapshot.loc[observed_snapshot['low_literacy_country'], 'd1_population_share'].sum() / d1_coverage
d1_lower_bound = observed_snapshot.loc[observed_snapshot['low_literacy_country'], 'd1_population_share'].sum()
d1_upper_bound = d1_lower_bound + (1 - d1_coverage)

snapshot_summary = pd.DataFrame([{
    'D1 population covered (%)': 100 * d1_coverage,
    'Median observation year': observed_snapshot.groupby('code')['observation_year'].first().median(),
    'Median observation age (years)': observed_snapshot.groupby('code')['observation_age'].first().median(),
    'Observed D1 share in countries below 50% literacy (%)': 100 * d1_low_share,
    'Lower bound for full D1 (%)': 100 * d1_lower_bound,
    'Upper bound for full D1 (%)': 100 * d1_upper_bound,
}])
display(snapshot_summary.round(1))

snapshot_correlations = pd.DataFrame([{
    'Countries': observed_snapshot['code'].nunique(),
    'Pearson r (D1 country share vs literacy deprivation)': observed_snapshot['d1_country_population_pct'].corr(100 - observed_snapshot['adult_literacy_pct']),
    'Spearman r (D1 country share vs literacy deprivation)': observed_snapshot['d1_country_population_pct'].rank().corr((100 - observed_snapshot['adult_literacy_pct']).rank()),
}])
display(snapshot_correlations.round(3))

,D1 population covered (%),Median observation year,Median observation age (years),Observed D1 share in countries below 50% literacy (%),Lower bound for full D1 (%),Upper bound for full D1 (%)
0,99.2,2021.0,3.0,13.4,13.3,14.1


,Countries,Pearson r (D1 country share vs literacy deprivation),Spearman r (D1 country share vs literacy deprivation)
0,170,0.679,0.692


## Recent country snapshot

The x-axis is the share of a country's population in the global D1 in 2024. The y-axis is that country's latest adult-literacy observation available by 2024. Points can therefore have different observation years; the tooltip shows the year and age of each observation. Correlations are country-level associations, not causal effects.

In [2]:
literacy_scatter = (
    alt.Chart(observed_snapshot.dropna(subset=['region']))
    .mark_circle(opacity=0.7, stroke='white', strokeWidth=0.5)
    .encode(
        x=alt.X('d1_country_population_pct:Q', title='Country population in global D1 (%)'),
        y=alt.Y('adult_literacy_pct:Q', title='Adult literacy rate (%)', scale=alt.Scale(domain=[0, 100])),
        color=alt.Color('region:N', title='World Bank region', scale=alt.Scale(domain=list(REGION_COLORS), range=list(REGION_COLORS.values()))),
        size=alt.Size('country_population:Q', legend=None, scale=alt.Scale(range=[40, 900])),
        tooltip=[
            alt.Tooltip('code:N', title='Country'),
            alt.Tooltip('region:N', title='Region'),
            alt.Tooltip('observation_year:Q', title='Literacy observation year', format='d'),
            alt.Tooltip('observation_age:Q', title='Observation age (years)', format='d'),
            alt.Tooltip('d1_country_population_pct:Q', title='Population in D1 (%)', format='.1f'),
            alt.Tooltip('adult_literacy_pct:Q', title='Adult literacy (%)', format='.1f'),
        ],
    )
    .properties(width=760, height=440, title='Latest adult literacy observation vs 2024 D1 concentration')
)
literacy_scatter

alt.Chart(...)

## Share of each decile in low-literacy countries

Each row reports the share of the 2024 global decile living in countries whose latest available national adult-literacy rate is below 50%. The lower/upper bounds treat all missing country population as respectively above/below the threshold. This does not estimate individual literacy within countries.

In [3]:
decile_snapshot = decile_distribution.merge(
    country_latest[['code', 'observation_year', 'adult_literacy_pct']],
    on='code',
    how='left',
)
decile_rows = []
for decile, group in decile_snapshot.groupby('decile'):
    observed = group.dropna(subset=['adult_literacy_pct'])
    total_population = group['decile_population'].sum()
    observed_population = observed['decile_population'].sum()
    low_population = observed.loc[
        observed['adult_literacy_pct'].lt(LOW_LITERACY_THRESHOLD), 'decile_population'
    ].sum()
    missing_population = total_population - observed_population
    decile_rows.append({
        'Decile': f'D{decile}',
        'Share in countries below 50% literacy (observed, %)': 100 * low_population / observed_population if observed_population else np.nan,
        'Population coverage (%)': 100 * observed_population / total_population,
        'Lower bound (% of full decile)': 100 * low_population / total_population,
        'Upper bound (% of full decile)': 100 * (low_population + missing_population) / total_population,
        'Median literacy observation year': observed.groupby('code')['observation_year'].first().median(),
    })
literacy_deciles = pd.DataFrame(decile_rows).set_index('Decile')
display(literacy_deciles.round(1))

,"Share in countries below 50% literacy (observed, %)",Population coverage (%),Lower bound (% of full decile),Upper bound (% of full decile),Median literacy observation year
Decile,,,,,
D1,13.4,99.2,13.3,14.1,2021.0
D2,6.1,99.7,6.0,6.3,2021.0
D3,2.7,99.8,2.7,3.0,2021.0
D4,1.8,99.7,1.8,2.1,2021.0
D5,1.4,99.5,1.4,1.9,2021.0
D6,1.0,99.1,0.9,1.8,2021.0
D7,0.5,97.5,0.5,3.0,2021.0
D8,0.2,90.4,0.2,9.8,2021.0
D9,0.1,67.5,0.1,32.6,2021.0


## Exact-year observations over time

Unlike electricity or internet use, literacy is measured sporadically through censuses and surveys. The table and chart below use only observations recorded in the same year as the D1 distribution. D1 coverage is included because the reporting-country mix changes over time; do not read this as a continuous global literacy trend.

In [4]:
exact_join = analysis['joined'].copy()
exact_join['literacy_deprivation_pct'] = 100 - exact_join['indicator_value']
year_rows = []
for year, group in exact_join.groupby('year'):
    countries = group['code'].nunique()
    d1_coverage_year = group['d1_population_share'].sum()
    below = group['indicator_value'].lt(LOW_LITERACY_THRESHOLD)
    d1_low_share = 100 * group.loc[below, 'd1_population_share'].sum() / d1_coverage_year
    world_low_share = 100 * group.loc[below, 'country_population'].sum() / group['country_population'].sum()
    year_rows.append({
        'Year': int(year),
        'Countries with data': countries,
        'D1 population covered (%)': 100 * d1_coverage_year,
        'Observed D1 in countries below 50% literacy (%)': d1_low_share,
        'World population in countries below 50% literacy (%)': world_low_share,
    })
exact_year_summary = pd.DataFrame(year_rows).sort_values('Year')
recent_exact_years = exact_year_summary.query('Year >= 2000 and `Countries with data` >= 40')
display(recent_exact_years.tail(15).round(1))

correlation_summary = analysis['correlations'].merge(
    exact_year_summary[['Year', 'D1 population covered (%)']],
    left_on='year',
    right_on='Year',
).query('observations >= 40')
display(correlation_summary[['year', 'pearson_r', 'spearman_r', 'observations', 'D1 population covered (%)']].query('year >= 2000').round(3))

trend_data = recent_exact_years.melt(
    id_vars=['Year', 'D1 population covered (%)', 'Countries with data'],
    value_vars=['Observed D1 in countries below 50% literacy (%)', 'World population in countries below 50% literacy (%)'],
    var_name='Group',
    value_name='Share (%)',
)
literacy_coverage_chart = (
    alt.Chart(trend_data)
    .mark_point(size=85, opacity=0.8)
    .encode(
        x=alt.X('Year:Q', title='Year', axis=alt.Axis(format='d')),
        y=alt.Y('Share (%):Q', title='Share living in countries below 50% literacy (%)'),
        color=alt.Color('Group:N', title=None),
        tooltip=[
            alt.Tooltip('Year:O', title='Year'),
            alt.Tooltip('Group:N', title='Group'),
            alt.Tooltip('Share (%):Q', title='Share (%)', format='.1f'),
            alt.Tooltip('D1 population covered (%):Q', title='D1 coverage (%)', format='.1f'),
            alt.Tooltip('Countries with data:Q', title='Countries with data'),
        ],
    )
    .properties(width=760, height=400, title='Literacy: reported-year observations (not a continuous global trend)')
)
literacy_coverage_chart

,Year,Countries with data,D1 population covered (%),Observed D1 in countries below 50% literacy (%),World population in countries below 50% literacy (%)
10,2000,44,53.5,1.7,1.4
20,2010,48,56.2,2.2,0.6
21,2011,53,34.7,8.4,1.9
22,2012,44,41.8,2.8,0.6
24,2014,45,30.3,14.3,4.4
25,2015,47,28.5,13.2,4.3
26,2016,45,41.2,12.4,6.9
27,2017,44,20.2,5.8,1.3
28,2018,49,50.2,9.0,3.1
29,2019,56,40.2,9.3,2.4


,year,pearson_r,spearman_r,observations,D1 population covered (%)
10,2000,0.356,0.410,44,53.466
20,2010,0.608,0.652,48,56.204
21,2011,0.758,0.658,53,34.706
22,2012,0.515,0.592,44,41.844
24,2014,0.664,0.666,45,30.320
25,2015,0.767,0.730,47,28.461
26,2016,0.601,0.623,45,41.215
27,2017,0.698,0.698,44,20.225
28,2018,0.539,0.675,49,50.182
29,2019,0.714,0.672,56,40.173


alt.Chart(...)

## Interpretation

The latest-country-observation snapshot has much better D1 coverage than an exact-year comparison, but countries' literacy observations are from different years. The exact-year series is temporally consistent but covers only part of the D1 and has irregular survey timing. Both are country-level measures, not literacy rates of D1 individuals.

Use the snapshot as exploratory context, report its observation-year and coverage caveats, and do not claim that low national literacy causes the income stagnation shown in the D1 income chart.